# Huấn luyện QLoRA (4-bit NF4) & Đánh giá VietOCR + Qwen2.5-1.5B/7B-Instruct
**Đề tài NCKH**: Nghiên cứu ứng dụng kỹ thuật OCR kết hợp Mô hình Ngôn ngữ Lớn (LLM) trong trích xuất thông tin từ hóa đơn bán lẻ tiếng Việt  
**Khung lý thuyết**: *Nested Learning: The Illusion of Deep Learning Architecture* (Behrouz et al., NeurIPS 2025) — Kết hợp Bộ nhớ Tần số cao CMS ($\mathcal{C}^{(1)}=1$), Tín hiệu Ngạc nhiên Cục bộ (LSS Self-Correction) và Tinh chỉnh QLoRA Tần số thấp ($\mathcal{C}^{(L)}=L$).

**Cấu trúc Kho Dữ liệu Chuẩn hóa (`Data_HoaDon_NCKH`)**:
- `01_MC_OCR_2021_Chuan_CuocThi/images/`: Các mẫu hóa đơn thực tế (`hoadon_thucte_001.jpg/.json/.txt` ...)
- `02_HoaDon_ThucTe_TuChup_300Anh/`: Kho lưu trữ 300 ảnh tự chụp của nhóm nghiên cứu
- `03_HoaDon_MoPhong_KiemThu_10Mau/`: Bộ 10 mẫu kiểm thử thách thức (`01_` đến `10_`)
- `04_DuLieu_DaChiaTap_Train_Val_Test/`: Chứa `splits_70_10_20.json`, `BoNho_CMS_FewShot_TrainOnly.jsonl`, `QLoRA_Train_Dataset.jsonl`, `QLoRA_Val_Dataset.jsonl`

## 1. Cài đặt Thư viện trên Google Colab / Kaggle T4 GPU (Bao gồm `vietocr`, `paddleocr`, `trl`, `bitsandbytes`)

In [ ]:
!pip install -q -U torch torchvision transformers peft bitsandbytes trl datasets accelerate jiwer pydantic vietocr paddlepaddle paddleocr

## 2. Cấu hình Đường dẫn tới Kho Dữ liệu `Data_HoaDon_NCKH` & Khởi tạo QLoRA (4-bit NF4, Double Quant, r=16, alpha=32)

In [ ]:
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

# Tự động dò tìm thư mục Data_HoaDon_NCKH hoặc fallback về data/processed
DATA_HUB_DIR = Path("../../Data_HoaDon_NCKH")
SPLIT_CMS_DIR = DATA_HUB_DIR / "04_DuLieu_DaChiaTap_Train_Val_Test"
if (SPLIT_CMS_DIR / "QLoRA_Train_Dataset.jsonl").exists():
    TRAIN_PATH = SPLIT_CMS_DIR / "QLoRA_Train_Dataset.jsonl"
    VAL_PATH = SPLIT_CMS_DIR / "QLoRA_Val_Dataset.jsonl"
    CMS_BANK_PATH = SPLIT_CMS_DIR / "BoNho_CMS_FewShot_TrainOnly.jsonl"
else:
    TRAIN_PATH = Path("../data/processed/qlora_train.jsonl")
    VAL_PATH = Path("../data/processed/qlora_val.jsonl")
    CMS_BANK_PATH = Path("../data/processed/cms_mcocr_train.jsonl")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Hoặc "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./qwen25_receipt_qlora"

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
print(f"Đang đọc dữ liệu huấn luyện từ: {TRAIN_PATH}")

## 3. Nạp Dữ liệu ChatML (`QLoRA_Train_Dataset.jsonl` & `QLoRA_Val_Dataset.jsonl`)

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_raw = load_jsonl(TRAIN_PATH)
val_raw = load_jsonl(VAL_PATH)
print(f"Số mẫu Train (Delta Rule Denoising): {len(train_raw)} | Số mẫu Validation: {len(val_raw)}")

## 4. Khởi tạo Model, Tokenizer và Huấn luyện `SFTTrainer`

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def to_chat_text(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = Dataset.from_list([to_chat_text(x) for x in train_raw])
val_ds = Dataset.from_list([to_chat_text(x) for x in val_raw])

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    optim="paged_adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

trainer.train()
model.save_pretrained(f"{OUTPUT_DIR}/final_lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_lora_adapter")

## 5. Đánh giá Toàn diện Sau Huấn luyện trên Cả 2 Bộ Dữ liệu (`01_MC_OCR_2021_Chuan_CuocThi` & `03_HoaDon_MoPhong_KiemThu_10Mau`)

In [ ]:
import sys
sys.path.insert(0, "../src")
from receipt_extraction.data.mcocr_loader import load_dataset_records
from receipt_extraction.evaluation.metrics import compute_field_metrics, compute_item_metrics
from receipt_extraction.extraction.llm_extractor import RECEIPT_EXTRACTION_SYSTEM_PROMPT, _extract_json_from_response
from receipt_extraction.ocr.factory import get_ocr_engine
from receipt_extraction.schemas.receipt import ReceiptData

# Khởi tạo VietOCR Engine thật trên GPU T4
vietocr_engine = get_ocr_engine("vietocr")
print("VietOCR status:", vietocr_engine.is_available())

test_dir = DATA_HUB_DIR / "01_MC_OCR_2021_Chuan_CuocThi" / "images"
if not test_dir.exists():
    test_dir = Path("../data/raw/real_receipts")

test_records = load_dataset_records(test_dir, split="test")
preds, gts = [], []

for rec in test_records:
    ocr_res = vietocr_engine.recognize(rec.image_path) if vietocr_engine.is_available()[0] else None
    ocr_text = (ocr_res.raw_text if ocr_res and ocr_res.is_success else rec.gt_ocr_text)
    messages = [
        {"role": "system", "content": RECEIPT_EXTRACTION_SYSTEM_PROMPT},
        {"role": "user", "content": f"Văn bản OCR hóa đơn:\n```text\n{ocr_text}\n```\nHãy trích xuất dữ liệu cấu trúc JSON 12 trường."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    gen_text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    parsed = _extract_json_from_response(gen_text)
    preds.append(ReceiptData(**parsed))
    gts.append(rec.receipt_data)

f12 = compute_field_metrics(preds, gts, mode="12_fields")
f4 = compute_field_metrics(preds, gts, mode="mcocr_4_fields")
items_m = compute_item_metrics(preds, gts)
print(f"Test 12-Field Micro-F1: {f12['micro_f1']:.4f} | 4-Field MC-OCR Micro-F1: {f4['micro_f1']:.4f} | Item Line-F1: {items_m['line_f1']:.4f}")